- deploy dspy program to mlflow server
- https://dspy.ai/tutorials/deployment/

In [ ]:
import dspy
import mlflow

mlflow.set_tracking_uri("http://127.0.0.1:5000/")
mlflow.set_experiment("deploy_dspy_program")

lm = dspy.LM(
    'ollama_chat/tiger-gemma2',
    api_base='http://localhost:11434',
)
# lm = dspy.LM("openai/gpt-4o-mini")

# dspy.configure(lm=lm)
dspy.settings.configure(lm=lm)

class MyProgram(dspy.Module):
    def __init__(self):
        super().__init__()
        self.cot = dspy.ChainOfThought("question -> answer")

    def forward(self, messages):
        return self.cot(question=messages[0]["content"])

dspy_program = MyProgram()

with mlflow.start_run():
    mlflow.dspy.log_model(
        dspy_program,
        "dspy_program",
        input_example={"messages": [{"role": "user", "content": "What is LLM agent?"}]},
        task="llm/v1/chat",
    )

MlflowException: API request to http://127.0.0.1:5000/api/2.0/mlflow/experiments/get-by-name failed with exception HTTPConnectionPool(host='127.0.0.1', port=5000): Max retries exceeded with url: /api/2.0/mlflow/experiments/get-by-name?experiment_name=deploy_dspy_program (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x74a4250baed0>: Failed to establish a new connection: [Errno 111] Connection refused'))